# 04 — SFT (arm A)

Full fine-tune of **Qwen3-0.6B-Base** on the cleaned SFT set (122,416 examples) with TRL's `SFTTrainer`.

- Loss on the **completion only** (the solution), never on the prompt
- Same prompt format as eval (`posttrain/prompts.py`), answer ends with `<|im_end|>`
- Checkpoints pushed to a **private Hub model repo** as they're saved (the Space's disk is temporary)
- Wall-clock time recorded for the paper's GPU-hour accounting

**Run order:** `SMOKE_TEST = True` first (a few minutes) → check the verification cell → set `SMOKE_TEST = False`, restart kernel, run the full training.

Hardware: A100 large (80 GB). Not used here: vLLM (no generation during SFT).

## 0. Imports + versions

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # reduces memory fragmentation

In [2]:
import os, sys, time, json
sys.path.insert(0, ".")

import torch, transformers, trl, datasets
from datasets import load_dataset
from transformers import AutoTokenizer
from trl import SFTConfig, SFTTrainer

from posttrain.prompts import to_prompt_completion, IM_END

print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| trl", trl.__version__, "| datasets", datasets.__version__)
print("GPU:", torch.cuda.get_device_name(0),
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")

torch 2.14.0+cu130 | transformers 5.22.2 | trl 1.18.1 | datasets 5.2.0
GPU: NVIDIA A10G 24 GB


## 1. Config — the only cell to edit

In [3]:
SMOKE_TEST = True            # True: ~2k examples, ~50 steps, nothing pushed. Then False for the real run.

MODEL_ID   = "Qwen/Qwen3-0.6B-Base"
HUB_ID     = "Maci10/dpo-king-sft"          # private model repo for checkpoints
OUTPUT_DIR = "checkpoints/sft"
SEED       = 42

# --- training hyperparameters (write these into the paper) ---
LEARNING_RATE   = 1e-5       # full fine-tune of a small model: 1e-5 .. 2e-5 is the usual range
NUM_EPOCHS      = 2          # ~122k examples; 2 epochs, pick the best checkpoint by validation
PER_DEVICE_BS   = 4         # sequences per step on the GPU
GRAD_ACCUM      = 16          # effective batch = 4 * 16 = 64 sequences
MAX_LENGTH      = 1024       # matches the length filter in notebook 01
WARMUP_RATIO    = 0.03
LR_SCHEDULER    = "cosine"
WEIGHT_DECAY    = 0.0

USE_WANDB = True             # set False if you didn't run `wandb login`
RUN_NAME  = "armA_sft_smoke" if SMOKE_TEST else "armA_sft"

## 2. Data
Rebuild `prompt` / `completion` from `problem` / `solution` with the **current** `prompts.py`,
so training always uses exactly the format eval uses. Validation = the 502 held-out MATH-train problems
(we track validation **loss** here; validation **accuracy** comes later with `03_eval`).

In [6]:
train_ds = load_dataset("Maci10/dpo-king-sft_all", split="train")
val_ds   = load_dataset("Maci10/dpo-king-math_val", split="train")

keep = ["prompt", "completion"]
train_ds = train_ds.map(to_prompt_completion)
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])
val_ds   = val_ds.map(to_prompt_completion)
val_ds   = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep])

if SMOKE_TEST:
    train_ds = train_ds.shuffle(seed=SEED).select(range(2000))
    val_ds   = val_ds.select(range(100))

print(len(train_ds), "train |", len(val_ds), "val")
print(train_ds[0]["prompt"] + train_ds[0]["completion"])

2000 train | 100 val
<|im_start|>system
Please reason step by step, and put your final answer within \boxed{}.<|im_end|>
<|im_start|>user
Let $f(x) = 2x - 3$ and $g(x) = x + 1$. What is the value of $f(1 + g(2))$?<|im_end|>
<|im_start|>assistant
First, we evaluate $g(2)$ by substituting $2$ into the function $g(x)$, so $g(2) = 2 + 1 = 3$.
Next, we substitute $3$ into the function $f(x)$, so $f(1 + g(2)) = f(1 + 3) = f(4)$.
Finally, we substitute $4$ into the function $f(x)$, so $f(4) = 2(4) - 3 = 8 - 3 = \boxed{5}$.<|im_end|>


- What’s good: the formatting is exactly right. System prompt, user question, the assistant’s step-by-step solution with \boxed{8}, and a single <|im_end|> at the end. That’s the format the model will learn.

- also the number for our test and val set both are the lens we set them to 

# 3. Tokienizer

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("eos:", tokenizer.eos_token, "|pad:", tokenizer.pad_token,
    "| <|im_end|> id:", tokenizer.convert_tokens_to_ids(IM_END))

eos: <|endoftext|> |pad: <|endoftext|> | <|im_end|> id: 151645


## 4. Trainer
Key choices:
- `completion_only_loss=True` → loss only on the solution tokens
- `eos_token="<|im_end|>"` → the model's end-of-answer token (TRL's docs recommend this for Qwen base models)
- `push_to_hub` + `hub_strategy="every_save"` → each checkpoint is uploaded as it's saved

In [8]:
steps_per_epoch = len(train_ds) // (PER_DEVICE_BS * GRAD_ACCUM) # per batch size * grad accum
save_every = 50 if SMOKE_TEST else max(steps_per_epoch // 4, 50) # 4 checkpoints per epoch 
# epoch = An epoch isn’t “how many training runs”. An epoch is one full pass through all the training data. With NUM_EPOCHS = 2, the model sees every one of the 122,416 examples twice, all within a single training run.

config = SFTConfig(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
    seed=SEED,
    # data / loss
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    eos_token=IM_END,
    packing=False,
    # optimisation
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    max_steps=50 if SMOKE_TEST else -1,
    per_device_train_batch_size=PER_DEVICE_BS,
    per_device_eval_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    bf16=True,
    gradient_checkpointing=False,      
    model_init_kwargs={"dtype": torch.bfloat16},
    # logging / eval / saving
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=save_every,
    save_strategy="steps",
    save_steps=save_every,
    save_total_limit=None,             # keep all checkpoints (needed for picking the best + DPO dynamics)
    report_to="wandb" if USE_WANDB else "none",
    # Hub
    push_to_hub=not SMOKE_TEST,
    hub_model_id=HUB_ID,
    hub_private_repo=True,
    hub_strategy="every_save",
)

trainer = SFTTrainer(
    model=MODEL_ID,
    args=config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)
print("steps/epoch:", steps_per_epoch, "| save/eval every:", save_every, "steps")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

steps/epoch: 31 | save/eval every: 50 steps


## 5. Verify one tokenized example BEFORE training (important)
Checks that (a) the sequence ends with exactly **one** `<|im_end|>`, and (b) the loss mask covers the
solution but not the prompt. If either check fails, stop and paste the output.

In [9]:
ex = trainer.train_dataset[0]
print("columns", trainer.train_dataset.column_names)

ids = ex["input_ids"]
mask = ex.get("completion_mask")
im_end_id = tokenizer.convert_tokens_to_ids(IM_END)

print("\n--- last 15 tokens ---")
print([tokenizer.decode([t]) for t in ids[-15:]])
n_end = sum(1 for t in ids[-3:] if t == im_end_id)
print("<|im_end|> in last 3 tokens:", n_end, "(want 1)")

if mask is not None:
    trained = tokenizer.decode([t for t, m in zip(ids, mask) if m])
    ignored = tokenizer.decode([t for t, m in zip(ids, mask) if not m])
    print("\n--- NOT trained on (prompt) ---\n", ignored[-200:])
    print("\n--- trained on (completion), start ---\n", trained[:200])
    print("\n--- trained on (completion), end ---\n", trained[-120:])
    print(f"\ncompletion tokens: {sum(mask)} / {len(ids)}")
else:
    print("No completion_mask column — paste this output before training.")

columns ['prompt', 'completion', 'input_ids', 'labels']

--- last 15 tokens ---
['3', ' =', ' ', '8', ' -', ' ', '3', ' =', ' \\', 'boxed', '{', '5', '}', '$.', '<|im_end|>']
<|im_end|> in last 3 tokens: 1 (want 1)
No completion_mask column — paste this output before training.


In [10]:
ex = trainer.train_dataset[0]
ids, labels = ex["input_ids"], ex["labels"]

trained = [t for t, l in zip(ids, labels) if l != -100]
ignored = [t for t, l in zip(ids, labels) if l == -100]

print("--- NOT trained on (prompt), end ---\n", tokenizer.decode(ignored)[-200:])
print("\n--- trained on (completion), start ---\n", tokenizer.decode(trained)[:200])
print("\n--- trained on (completion), end ---\n", tokenizer.decode(trained)[-80:])
print(f"\ntrained tokens: {len(trained)} / {len(ids)}")
print("labels match input_ids where trained:", all(l == t for t, l in zip(ids, labels) if l != -100))

--- NOT trained on (prompt), end ---
  reason step by step, and put your final answer within \boxed{}.<|im_end|>
<|im_start|>user
Let $f(x) = 2x - 3$ and $g(x) = x + 1$. What is the value of $f(1 + g(2))$?<|im_end|>
<|im_start|>assistant


--- trained on (completion), start ---
 First, we evaluate $g(2)$ by substituting $2$ into the function $g(x)$, so $g(2) = 2 + 1 = 3$.
Next, we substitute $3$ into the function $f(x)$, so $f(1 + g(2)) = f(1 + 3) = f(4)$.
Finally, we substit

--- trained on (completion), end ---
 4$ into the function $f(x)$, so $f(4) = 2(4) - 3 = 8 - 3 = \boxed{5}$.<|im_end|>

trained tokens: 123 / 191
labels match input_ids where trained: True


## 6. Train (records wall-clock time for GPU-hour accounting)

In [11]:
t0 = time.time()
train_result = trainer.train()
train_secs = time.time() - t0

gpu_hours = train_secs / 3600
print(f"Training time: {train_secs/60:.1f} min = {gpu_hours:.3f} GPU-hours on {torch.cuda.get_device_name(0)}")
print(train_result.metrics)

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /home/user/.netrc.


wandb: Currently logged in as: macisekander10 (macisekander10-columbia-university) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,0.349059,0.830378,0.627535,0.783337,710943.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training time: 3.5 min = 0.058 GPU-hours on NVIDIA A10G
{'train_runtime': 209.7282, 'train_samples_per_second': 15.258, 'train_steps_per_second': 0.238, 'total_flos': 2846416984080384.0, 'train_loss': 0.36931212425231935, 'epoch': 1.576}


In [12]:
trainer.save_model(OUTPUT_DIR)          # final weights + tokenizer
summary = {
    "arm": "armA_sft", "base_model": MODEL_ID, "smoke_test": SMOKE_TEST,
    "n_train": len(train_ds), "n_val": len(val_ds),
    "learning_rate": LEARNING_RATE, "epochs": NUM_EPOCHS, "per_device_bs": PER_DEVICE_BS,
    "grad_accum": GRAD_ACCUM, "max_length": MAX_LENGTH, "warmup_ratio": WARMUP_RATIO,
    "scheduler": LR_SCHEDULER, "seed": SEED,
    "gpu": torch.cuda.get_device_name(0), "train_seconds": round(train_secs, 1),
    "gpu_hours": round(gpu_hours, 4), "train_metrics": train_result.metrics,
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__, "trl": trl.__version__},
}
with open(os.path.join(OUTPUT_DIR, "sft_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

if not SMOKE_TEST:
    trainer.push_to_hub(commit_message="Final SFT model (arm A)")
    print("Pushed to", HUB_ID)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "arm": "armA_sft",
  "base_model": "Qwen/Qwen3-0.6B-Base",
  "smoke_test": true,
  "n_train": 2000,
  "n_val": 100,
  "learning_rate": 1e-05,
  "epochs": 2,
  "per_device_bs": 4,
  "grad_accum": 16,
  "max_length": 1024,
  "warmup_ratio": 0.03,
  "scheduler": "cosine",
  "seed": 42,
  "gpu": "NVIDIA A10G",
  "train_seconds": 209.9,
  "gpu_hours": 0.0583,
  "train_metrics": {
    "train_runtime": 209.7282,
    "train_samples_per_second": 15.258,
    "train_steps_per_second": 0.238,
    "total_flos": 2846416984080384.0,
    "train_loss": 0.36931212425231935,
    "epoch": 1.576
  },
  "versions": {
    "torch": "2.14.0+cu130",
    "transformers": "5.22.2",
    "trl": "1.18.1"
  }
}
